In [1]:
# --- Célula 1: Coleta e Diagnóstico Inicial dos Dados Brutos ---

# Instala as bibliotecas necessárias
!pip install -q yfinance pandas_datareader

import yfinance as yf
import pandas as pd
from datetime import datetime
import pandas_datareader.data as web
import warnings
warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO DAS DATAS ---
end_date = '2025-08-31'
start_date = (pd.to_datetime(end_date) - pd.DateOffset(years=5)).strftime('%Y-%m-%d')
print(f"A recolher dados de {start_date} até {end_date}...")

# --- ETAPA 1: BAIXAR DADOS DE MERCADO DO YAHOO FINANCE ---
tickers = {
    'aluminum': 'ALI=F', 'copper': 'HG=F', 'zinc': 'ZN=F',
    'tin': 'SN=F', 'lead': 'PB=F', 'steel': 'S00=F',
    'oil': 'BZ=F', 'gas': 'NG=F', 'dxy': 'DX-Y.NYB',
    'usdcny': 'USDCNY=X', 'eurusd': 'EURUSD=X', 'brlusd': 'BRL=X',
    'msci_em': 'EEM', 'xlb': 'XLB', 'pick': 'PICK', 'fxi': 'FXI',
    'alcoa': 'AA', 'century_aluminum': 'CENX', 'hindalco': 'HINDALCO.NS'
}
# Baixa os preços de fecho
market_data = yf.download(list(tickers.values()), start=start_date, end=end_date)['Close']
market_data.columns = list(tickers.keys())
market_data = market_data.reset_index().rename(columns={'Date': 'date'})
print("Dados de mercado do Yahoo Finance recolhidos.")

# --- ETAPA 2: BAIXAR DADOS MACROECONÓMICOS DO FRED ---
fred_series = {
    'us_cpi': 'CPIAUCSL',
    'us_ind_production': 'INDPRO',
    'china_pmi': 'CHNPMI',
    # Adicione outros códigos do FRED aqui se desejar
}
fred_data = pd.DataFrame()
for name, code in fred_series.items():
    try:
        serie = web.DataReader(code, 'fred', start_date, end_date)
        serie.columns = [name]
        if fred_data.empty:
            fred_data = serie
        else:
            fred_data = fred_data.join(serie, how='outer')
    except Exception as e:
        print(f"AVISO: Falhou a recolha de '{name}' do FRED: {e}")
if not fred_data.empty:
    fred_data = fred_data.reset_index().rename(columns={"DATE": "date"})
    print("Dados macroeconómicos do FRED recolhidos.")
    # --- ETAPA 3: UNIR DADOS BRUTOS ---
    df_raw = pd.merge(market_data, fred_data, on="date", how="left")
else:
    df_raw = market_data

# Remove colunas que possam ter falhado o download e ficado totalmente nulas
df_raw.dropna(axis=1, how='all', inplace=True)

# ======================== DIAGNÓSTICO DE VITALIDADE DAS SÉRIES (antes do fill/interp) ========================
print("\n### Diagnóstico: Última data COM VALOR REAL (não-NaN) para cada coluna bruta ###")
for col in df_raw.columns:
    if col != 'date':
        dt_ult = df_raw.loc[df_raw[col].notna(), 'date'].max()
        qtd_nulos_final = df_raw[col].tail(30).isna().sum()  # NaNs nas últimas 30 datas
        print(f"{col:25s}  |  Último dado real: {dt_ult}  |  NaNs últimas 30 datas: {qtd_nulos_final}")

# =====================================================================================

# Preenche os valores nulos (ex: fins de semana) usando interpolação linear
df_raw.interpolate(method='linear', limit_direction='both', inplace=True)
df_raw = df_raw.sort_values('date').reset_index(drop=True)

# Padroniza as colunas de metais para USD/tonelada para consistência
# (Esta lógica foi extraída do seu notebook Aluminium_realista.ipynb)
if 'aluminum' in df_raw.columns: df_raw['aluminum_usdton'] = df_raw['aluminum'] * 100
if 'copper' in df_raw.columns: df_raw['copper_usdton'] = (df_raw['copper'] / 100) * 2204.62
if 'zinc' in df_raw.columns: df_raw['zinc_usdton'] = df_raw['zinc'] * 100
if 'tin' in df_raw.columns: df_raw['tin_usdton'] = df_raw['tin'] * 100
if 'lead' in df_raw.columns: df_raw['lead_usdton'] = df_raw['lead'] * 100

print("\nPipeline de Coleta de Dados Brutos concluído!")
print(f"DataFrame bruto final com {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas.")
print("Últimos 5 registos de dados:")
display(df_raw.tail())


A recolher dados de 2020-08-31 até 2025-08-31...


[**********************84%***************        ]  16 of 19 completedHTTP Error 404: 
HTTP Error 404: 
[*********************100%***********************]  19 of 19 completed

3 Failed downloads:
['PB=F', 'S00=F', 'SN=F']: YFTzMissingError('possibly delisted; no timezone found')


Dados de mercado do Yahoo Finance recolhidos.
AVISO: Falhou a recolha de 'china_pmi' do FRED: Unable to read URL: https://fred.stlouisfed.org/graph/fredgraph.csv?id=CHNPMI
Response Text:
b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n    <meta charset="utf-8">\r\n    <meta http-equiv="X-UA-Compatible" content="IE=edge">\r\n    <meta name="viewport" content="width=device-width, initial-scale=1">\r\n    <title>Error - St. Louis Fed</title>\r\n    <meta name="description" content="">\r\n    <meta name="keywords" content="">    \r\n    <link rel="stylesheet" type="text/css" href="/assets/bootstrap/dist/css/bootstrap.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/css/home.min.css?1553087253">\r\n    <link rel="stylesheet" type="text/css" href="/assets/fontawesome-free/css/all.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/assets/select2/dist/css/select2.min.css">\r\n    <style>p {\r\n        margin-bottom: 1.5em;\r\n    }</style>\r\n</head>\r\n<body>\r\n<

,date,aluminum,copper,zinc,tin,lead,steel,oil,gas,dxy,...,alcoa,century_aluminum,hindalco,us_cpi,us_ind_production,aluminum_usdton,copper_usdton,zinc_usdton,tin_usdton,lead_usdton
1301,2025-08-25,31.799999,2527.00,5.4213,68.800003,22.290001,98.430000,50.340000,1.170960,39.139999,...,7.1675,91.669998,111.968750,323.364,103.9867,3179.999924,55710.74740,542.129993,6880.000305,2229.000092
1302,2025-08-26,31.950001,2541.25,5.4097,67.220001,22.450001,98.230003,50.209999,1.161845,39.220001,...,7.1510,91.919998,112.203125,323.364,103.9867,3195.000076,56024.90575,540.969992,6722.000122,2245.000076
1303,2025-08-27,31.799999,2506.00,5.4312,68.050003,22.700001,98.230003,49.910000,1.163941,38.220001,...,7.1520,92.150002,112.406250,323.364,103.9867,3179.999924,55247.77720,543.120003,6805.000305,2270.000076
1304,2025-08-28,31.920000,2516.25,5.4177,68.620003,22.379999,97.809998,50.099998,1.164795,38.560001,...,7.1530,92.150002,112.531250,323.364,103.9867,3192.000008,55473.75075,541.769981,6862.000275,2237.999916
1305,2025-08-29,32.189999,2514.75,5.4130,68.120003,22.330000,97.769997,49.860001,1.168156,38.910000,...,7.1530,92.279999,112.453125,323.364,103.9867,3218.999863,55440.68145,541.300011,6812.000275,2232.999992


In [3]:
import numpy as np
# --- Célula 2: Engenharia de Features Completa ---

if 'df_raw' in locals():
    df = df_raw.copy()

    # --- ETAPA 1: DEFINIÇÃO DE VARIÁVEIS E FUNÇÕES ---
    
    # Lista de todos os ativos para criar features
    ativos = [col for col in df.columns if col not in ['date']]
    
    # Função para criar os alvos (targets) dinâmicos
    def add_dynamic_targets(df_in, price_column="aluminum_usdton", horizon=1):
        df_in[f'future_{horizon}d'] = df_in[price_column].shift(-horizon)
        df_in[f'ret_{horizon}d'] = (df_in[f'future_{horizon}d'] - df_in[price_column]) / df_in[price_column]
        
        def classify_bin(x):
            if x > 0.10: return "Sobe muito"
            elif x > 0.03: return "Sobe"
            elif x >= -0.03: return "Na mesma"
            elif x > -0.10: return "Cai"
            else: return "Cai muito"
        
        df_in[f'bin_{horizon}d'] = df_in[f'ret_{horizon}d'].apply(classify_bin)
        return df_in

    # --- ETAPA 2: CRIAÇÃO DAS FEATURES ---

    # 2.1 - Alvos dinâmicos
    horizons = [1, 5, 30, 90]
    for h in horizons:
        df = add_dynamic_targets(df, price_column="aluminum_usdton", horizon=h)

    # 2.2 - Retornos e Log-Retornos
    for ativo in ativos:
        df[f'{ativo}_ret1'] = df[ativo].pct_change()
        df[f'{ativo}_logret1'] = np.log(df[ativo] / df[ativo].shift(1))

    # 2.3 - Rolling Windows e Lags
    rolling_windows = [5, 10, 21]
    lags = [1, 2, 3, 5, 10, 21]
    for ativo in ativos:
        for win in rolling_windows:
            df[f'{ativo}_ma{win}'] = df[ativo].rolling(window=win).mean()
            df[f'{ativo}_std{win}'] = df[ativo].rolling(window=win).std()
            df[f'{ativo}_max{win}'] = df[ativo].rolling(window=win).max()
            df[f'{ativo}_min{win}'] = df[ativo].rolling(window=win).min()
        for lag in lags:
            df[f'{ativo}_lag{lag}'] = df[ativo].shift(lag)

    # 2.4 - Spreads e Ratios (relações entre mercados)
    df['copper_minus_aluminum'] = df.get('copper_usdton', pd.Series(0)) - df.get('aluminum_usdton', pd.Series(0))
    df['oil_minus_gas'] = df.get('oil', pd.Series(0)) - df.get('gas', pd.Series(0))
    df['copper_div_aluminum'] = df.get('copper_usdton', pd.Series(0)) / df.get('aluminum_usdton', pd.Series(0))

    # 2.5 - Indicadores Técnicos específicos do Alumínio
    for win in rolling_windows:
        df[f'alum_zscore_ma{win}'] = (df['aluminum_usdton'] - df[f'aluminum_usdton_ma{win}']) / df[f'aluminum_usdton_std{win}']
        df[f'alum_above_ma{win}'] = (df['aluminum_usdton'] > df[f'aluminum_usdton_ma{win}']).astype(int)
    
    df['alum_cross_ma5_ma21'] = (df['aluminum_usdton_ma5'] > df['aluminum_usdton_ma21']).astype(int)
    df['alum_oversold'] = (df['alum_zscore_ma21'] < -1.5).astype(int)
    df['alum_overbought'] = (df['alum_zscore_ma21'] > 1.5).astype(int)

    # 2.6 - Features de Calendário
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['weekday'] = df['date'].dt.weekday
    df['end_of_month'] = df['date'].dt.is_month_end.astype(int)
    
    # --- ETAPA 3: LIMPEZA FINAL ---
    
    # Remove colunas duplicadas que podem ter sido criadas
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Substitui valores infinitos (de divisões por zero) por NaN e depois preenche
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(method='ffill', inplace=True)
    df.dropna(inplace=True) # Remove as primeiras linhas que ainda têm NaNs dos lags/rolling
    df = df.reset_index(drop=True)

    # Salva o DataFrame final
    output_filename = 'aluminium_full_featured_retrained.csv'
    df.to_csv(output_filename, index=False)

    print(f"\nPipeline de Engenharia de Features concluído!")
    print(f"DataFrame final com {df.shape[0]} linhas e {df.shape[1]} colunas.")
    print(f"Ficheiro salvo como: '{output_filename}'")
    display(df.head())

else:
    print("ERRO: DataFrame 'df_raw' não foi criado. Execute a Célula 1 primeiro.")


Pipeline de Engenharia de Features concluído!
DataFrame final com 1285 linhas e 512 colunas.
Ficheiro salvo como: 'aluminium_full_featured_retrained.csv'


,date,aluminum,copper,zinc,tin,lead,steel,oil,gas,dxy,...,alum_above_ma10,alum_zscore_ma21,alum_above_ma21,alum_cross_ma5_ma21,alum_oversold,alum_overbought,month,quarter,weekday,end_of_month
0,2020-09-29,11.176966,1786.5,5.6602,41.029999,7.13,93.889999,38.761597,1.167883,36.600735,...,0,-1.489122,0,0,0,0,9,3,1,0
1,2020-09-30,11.138659,1778.5,5.6305,40.950001,7.12,93.889999,39.386932,1.174205,37.338619,...,0,-1.394398,0,0,0,0,9,3,2,1
2,2020-10-01,10.908797,1743.0,5.6084,40.930000,6.94,93.709999,39.753201,1.172608,37.792007,...,0,-1.452225,0,0,0,0,10,4,3,0
3,2020-10-02,11.320629,1781.0,5.6420,39.270000,7.13,93.839996,39.297592,1.174495,37.365284,...,1,-0.976865,0,0,0,0,10,4,4,0
4,2020-10-05,11.425982,1783.5,5.6816,41.290001,7.15,93.510002,39.824661,1.172044,37.507526,...,1,-0.799115,0,0,0,0,10,4,0,0
